In [ ]:
!pip install -r requirements.txt

In [ ]:
#import packages
import tqdm
from tqdm import tqdm
from tqdm.notebook import tqdm_notebook
import math

import numpy as np
from numpy import array

from numpy.linalg import norm
import pickle
import pandas as pd
import torch

import math
from pathlib import Path
import os.path
import ipywidgets
import anywidget
"""
import gpytorch
from torch.nn import Linear
from gpytorch.means import ConstantMean, LinearMean
from gpytorch.kernels import MaternKernel, ScaleKernel
from gpytorch.kernels import RBFKernel
from gpytorch.variational import VariationalStrategy, CholeskyVariationalDistribution, \
    LMCVariationalStrategy
from gpytorch.variational import MeanFieldVariationalDistribution
from gpytorch.distributions import MultivariateNormal
from gpytorch.models.deep_gps import DeepGPLayer, DeepGP
from gpytorch.models.deep_gps.dspp import DSPPLayer, DSPP
from gpytorch.mlls import DeepApproximateMLL, VariationalELBO
from gpytorch.likelihoods import MultitaskGaussianLikelihood
from gpytorch.mlls import AddedLossTerm
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import DeepPredictiveLogLikelihood
import gpytorch.settings as settings
import tensorflow as tf
print(tf.__version__)
from tensorflow import keras
from tensorflow.keras import layers
from keras.utils import plot_model
from tensorflow.keras.optimizers import RMSprop
#import tensorflow_datasets as tfds
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras.layers import Dropout
from keras.optimizers import SGD
# Make plots inline
"""
from snowflake import telemetry
from scipy.spatial import distance_matrix
from scipy.cluster.vq import kmeans2
import matplotlib
from matplotlib import pyplot as plt
#%matplotlib inline

In [ ]:
%run ./PY_INITIALIZE.ipynb

In [ ]:
# Import python packages
import logging

# set up the logger
logger_name = 'process_logger'
logger = logging.getLogger(logger_name)
logger.setLevel(logging.INFO)

In [ ]:
import snowflake.snowpark.modin.plugin
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import modin.pandas as pd

In [ ]:
clmsptData = pd.read_snowflake(f"{database_name}.{schema_name}.FINAL_TABLE")
logger.info(f"The clmsptData is read into a dataframe")

In [ ]:
#clmsptData['INC_FROM'] = clmsptData['INC_FROM'].str.replace(r'[#*]', '-', regex=True)
clmsptData['INC_FROM'] = pd.to_datetime(clmsptData['INC_FROM'].str.strip('"'))
clmsptData['clm_yr'] = clmsptData['INC_FROM'].dt.year
clmsptData['clm_yr'] = clmsptData['clm_yr'].astype(int)
strt_yr = clmsptData['clm_yr'].min()
last_yr = clmsptData['clm_yr'].max()
clmsptData.drop(columns=['INC_FROM','INC_TO'], inplace=True)

logger.info(f"Extracted the first ({strt_yr}) and last ({last_yr}) claim year")

In [ ]:
def train_val_dataprep(tr_yr1, tr_yr2):
    yr_diff = (tr_yr2)-(tr_yr1)
    print(tr_yr2, tr_yr1,yr_diff)
    final_merge = pd.DataFrame()
    for i in range (yr_diff):
        yr1 = tr_yr1+i
        yr2 = tr_yr1+i+1
        print("Preparing data for "+str(yr1)+" and "+str(yr2))
        print(list(clmsptData.columns.values))
        df1 = clmsptData[clmsptData['clm_yr']==yr1]
        df2 = clmsptData[clmsptData['clm_yr']==yr2]
        #print(list(df1.columns.values), list(df2.columns.values))
        df1.groupby(['MBR_ID','ICD_CD','CPT_CD'])['PAID_AMT'].agg('sum')
        df2.groupby(['MBR_ID','ICD_CD','CPT_CD'])['PAID_AMT'].agg('sum')
        #print(list(df1.columns.values), list(df2.columns.values))
        merged_df = pd.merge(df2, df1, how='left', on = ['MBR_ID','ICDCD_NUMCODED','SUBCD_NBR'], suffixes=('_next', '_prev')) 
        merged_df['clm_diff_col']=merged_df['PAID_AMT_next']-merged_df['PAID_AMT_prev']
        valid_clm_diff = merged_df.dropna(subset=['clm_diff_col'], inplace=True)
        final_merge = pd.concat([final_merge,merged_df], ignore_index=True)
        print(f"lenghts: {len(merged_df)}, {len(final_merge)}")
    rename_dict = {'FST_NAME_prev':'FST_NAME','MDL_NAME_prev':'MDL_NAME',\
        'LST_NAME_prev':'LST_NAME','PT_AGE_prev':'PT_AGE', 'PT_ZIP_prev':'PT_ZIP',\
            'ICD_CD_prev':'ICD_CD', 'CPT_CD_prev':'CPT_CD',\
                'OMC_ICD_RISK_NBR_prev':'OMC_ICD_RISK_NBR', 'PAID_AMT_next':'PAID_AMT_nxtyr'}
    
    final_merge.rename(columns=rename_dict, inplace=True)
    final_merge = final_merge.drop(columns=final_merge.columns[final_merge.columns.str.endswith('_next')])
    colsorder = ["MBR_ID", 'FST_NAME', 'MDL_NAME', 'LST_NAME', 'PT_AGE', 'PT_ZIP',\
    'ICD_CD','CPT_CD','ICDCD_NUMCODED', 'OMC_ICD_RISK_NBR','SUBCD_NBR',\
        'PAID_AMT_prev', 'PAID_AMT_nxtyr', 'clm_diff_col']
    columns = list(final_merge.columns.values)
    logger.info(f"{columns}")
    final_merge = final_merge[colsorder]
    final_merge.drop_duplicates(subset=['MBR_ID', 'ICDCD_NUMCODED', 'SUBCD_NBR'], inplace=True)
    if tr_yr1 == strt_yr:
        stage = 'train'
    else:
        stage = 'validat'
    final_merge['OMC_ICD_RISK_NBR'] = final_merge['OMC_ICD_RISK_NBR'].astype(int)
    logger.info(f"The {stage}ing data dataframe columns are {columns}")
    final_merge.to_snowflake(f"{database_name}.{schema_name}.{stage}ing_Table", if_exists='replace', index=False)
    logging.info(f"{stage}ing data file written to database: {database_name} schema: {schema_name}")


In [ ]:
def prep_skew_data(x):
    pt = PowerTransformer(method='yeo-johnson', standardize=True)

    # Fit and transform the data
    # standardize=True normalizes output to mean=0 and variance=1
    transformed_data = pt.fit_transform(data)
    
    # Check the learned optimal lambda values for each column
    return transformed_data

In [ ]:
def xy_transform(xy_table):
    xy_table['PT_AGE_trsfm'] = pd.series(prep_skew_data(xy_table['PT_AGE'].to_numpy()))
    xy_table['PT_ZIP_trsfm'] = pd.series(prep_skew_data(xy_table['PT_ZIP'].to_numpy()))
    xy_table['ICDCD_NUMCODED_trsfm'] = pd.series(prep_skew_data(xy_table['ICDCD_NUMCODED'].to_numpy()))
    xy_table['SUBCD_NBR_trsfm'] = pd.series(prep_skew_data(xy_table['SUBCD_NBR'].to_numpy()))
    xy_table['OMC_ICD_RISK_NBR_trsfm'] = pd.series(prep_skew_data(xy_table['OMC_ICD_RISK_NBR'].to_numpy()))
    xy_table['CPT_trsfm'] = pd.series(prep_skew_data(xy_table['CPT'].to_numpy()))
    y = xy_table['clm_diff_col'].to_numpy()
    if y.std()==0:
        y_N = y
    else:
        y_N = (y-y.mean())/y.std()
    y_trueMean = y.mean()
    y_trueSD = y.SD()
    xy_table['y_trsfm'] = y_N
    
    return (xy_table, y_trueMean, y_trueSD)

In [ ]:
def get_xy_data(prepped_data):
    if(prepped_data.empty): 
        print("Dataframe for the claim difference range "+level+" is empty")
        null=1
        # Read the training data from the sf_2016_2017 dataframe
        x1 = {}
        x2 = {}
        x3 = {}
        x4 = {}
        x5 = {}
        y = {}
    else:
        # Read the training data from the sf_2016_2017 dataframe
        """
        x1 = np.vstack(prepped_data['PT_AGE_trsfm']).astype(int)
        x2 = np.vstack(prepped_data['PT_ZIP_trsfm']).astype(int)
        x3 = np.vstack(prepped_data['ICDCD_NUMCODED_trsfm']).astype(np.float32)
        x4 = np.vstack(prepped_data['OMC_ICD_RISK_NBR_trsfm']).astype(np.float32)
        x5 = np.vstack(prepped_data['SUBCD_NBR_trsfm']).astype(np.float32)
        y = np.vstack(prepped_data['y_trsfm']).astype(np.float32)
        """
        # Extract non transformed column values
        x1 = np.vstack(prepped_data['PT_AGE']).astype(int)
        x2 = np.vstack(prepped_data['PT_ZIP']).astype(int)
        x3 = np.vstack(prepped_data['ICDCD_NUMCODED']).astype(np.float32)
        x4 = np.vstack(prepped_data['OMC_ICD_RISK_NBR']).astype(np.float32)
        x5 = np.vstack(prepped_data['SUBCD_NBR']).astype(np.float32)
        y = np.vstack(prepped_data['CLM_DIFF_COL']).astype(np.float32)
    
    return x1, x2, x3,x4,x5,y

In [ ]:
def get_tensor_dataset(x1,x2, x3, x4, x5,y):
    # COnverting training data and testing data to torch tensor
    x_train = np.column_stack((x1,x2, x3, x4, x5))
    #y_train = nxt_yr_diff
    y_train = y
    y_train = np.reshape(y,[-1])
    train_x = torch.tensor(x_train, dtype=torch.float)
    print(train_x.shape)
    print(train_x.shape[-1])
    
    
    train_y = torch.tensor(y_train, dtype=torch.float) 
    print(train_y.shape)
    
    
    val_x = train_x
    val_y = train_y
    return train_x, train_y, val_x, val_y

In [ ]:
stages = ['Train', 'Validat']
#stages = ['Validate', 'Test']

In [ ]:
from snowflake.ml.registry import Registry
import datetime
import snowflake.snowpark.modin.plugin

# Connect to your registry
reg = Registry(session=session, database_name=database_name, schema_name=schema_name)

# Generate a clean timestamp string (e.g., v2026_06_28_1703)
timestamp_version = datetime.datetime.now().strftime("v%Y_%m_%d_%H%M")


In [ ]:
from snowflake.ml.modeling.xgboost import XGBRegressor

In [ ]:
def update_version(name):
    last2 = name[-2:]
    last2 = int(last2)
    if int(last2) < 9:
        return "v0"+str(last2+1)
    else:
        return "v"+str(last2+1)

In [ ]:
from snowflake.ml.registry import Registry

def save_model(model, model_name, X_train, FeatureCols,TargetCol,rmse_metrics, overwrite):
    #Initialize the model list dataframe

    X_train = X_train._to_pandas()
    df_models = []
    filtered_models = []
    #make sure model_name is string
    model_name = str(model_name)
    # set up registry
    m = Registry(
        session=session, 
        database_name=database_name, 
        schema_name=model_schema
    )
    # 2. find if trained model exists
    df_models = m.show_models()
    if len(df_models)>0:
        filtered_models = df_models[df_models['name'].str.contains(model_name, case=False, na=False)]
    # update model name and version depending on filtered_models and overwrite
    if (len(filtered_models) > 0): # Implies XGB Regressor trained model is there
        
        models_sort = filtered_models.sort_values(by="created_on", ascending=False)
        last2 = models_sort.iloc[0]["name"][-2:]
        #print("Last 2 characters in latest:"+str(last2))
        try:
            version_no = int(last2)
            model_version = "v"+str(version_no)
            latest_model =  models_sort.iloc[0]["name"]
        except ValueError:
            print("Model name does not have version number in the last 2 characters")

        if overwrite:
            print(f"Overwrite:{overwrite}")
            model_name = latest_model
            m.delete_model(model_name)
            model_version = "v"+last2
        else:
            print(f"Overwrite:{overwrite}")
            model_version = update_version(latest_model)
            new_model_name = model_name+"_"+model_version
            model_name = new_model_name
    else:
        model_version = "v01"
        model_name = model_name+"_v01"
    
    # Define your artifact repository (e.g., your mapped PyPI registry)
    
    # Force convert to a clean, index-free Pandas DataFrame
    clean_sample_data = X_train.head(100).copy()
    if isinstance(clean_sample_data, pd.Series):
        clean_sample_data = clean_sample_data.to_frame()
    # making sure the sample input data has just the feature columns
    clean_sample_data = clean_sample_data[FeatureCols]
    # Create a signature specifying input features and output
    # Pass prediction array instead of X_train[TargetCol]
    sig = model_signature.infer_signature(
        X_train[FeatureCols], 
        model.predict(X_train[FeatureCols].head(1)) 
    )
    # 2. Save the model directly to ML schema
    mv = m.log_model(
        model=model,
        version_name=model_version,
        model_name=model_name,
        conda_dependencies=["xgboost", "scikit-learn"],
        signatures={"predict": sig},
        sample_input_data=X_train[FeatureCols].head(5), # Add this back
        target_platforms=["WAREHOUSE"],
        options={'relax_version': False},
        metrics = rmse_metrics
    )

    print(f"Model saved successfully as: {mv.model_name} version: {mv.version_name}")

    

In [ ]:
from snowflake.ml.registry import Registry

def delete_model():
    #Initialize the model list dataframe

    
    df_models = []
    filtered_models = []
    
    # set up registry
    m = Registry(
        session=session, 
        database_name=database_name, 
        schema_name=model_schema
    )
    # 2. find if trained model exists
    df_models = m.show_models()
    if len(df_models)>0:
        for index in range(len(df_models)):
            model_name =  df_models.iloc[index]["name"]
            m.delete_model(model_name)

delete_model()

In [ ]:
def retrieve_model(model_name):
    #Initialize the model list dataframe
    df_models = []
    filtered_models = []
    reg = Registry(
        session=session, 
        database_name=database_name, 
        schema_name=model_schema
    )
    df_models = reg.show_models()
    if len(df_models)>0:
        filtered_models = df_models[df_models['name'].str.contains(model_name, case=False, na=False)]

    if (len(filtered_models)>0):       
        models_sort = filtered_models.sort_values(by="created_on", ascending=False)
        last2 = models_sort.iloc[0]["name"][-2:]
        model_name = models_sort.iloc[0]["name"]
        model_version = "v"+str(last2)
        model = reg.get_model(model_name).version(model_version)

        
        return model
    else:
        print("Trained model not available")
        return null
    

In [ ]:
def identify_non_numeric_cols(modinpd_df, FeatureCols):
    # Identify columns with mixed or non-numeric types
    for col in FeatureCols:
        # Try converting to numeric, coercing errors to NaN
        converted = pd.to_numeric(modinpd_df[col], errors='coerce')
        
        # Check if the conversion resulted in new NaNs compared to original non-NA values
        non_numeric_rows = modinpd_df[col][converted.isna() & modinpd_df[col].notna()]
        
        if not non_numeric_rows.empty:
            print(f"Non-numeric values found in column: '{col}'")
            print(non_numeric_rows.head())


In [ ]:
ColDtype_dict = {"MBR_ID": 'string',"FST_NAME":'string',"MDL_NAME":'string',\
    "LST_NAME":'string',"PT_AGE":"int64","PT_ZIP":"int64","ICD_CD":'string',"CPT_CD":'string',\
        "ICDCD_NUMCODED":'float64', "OMC_ICD_RISK_NBR":'int64',"SUBCD_NBR":'float64',\
            "PAID_AMT_PREV":'float64',"PAID_AMT_NXTYR":'float64',"CLM_DIFF_COL":'float64'}
    

In [ ]:
def load_table(StoredTable, CleanedTable):
        
        sql_query = f"""CREATE or REPLACE TABLE {CleanedTable} AS
            SELECT 
            TO_VARIANT(MBR_ID) AS MBR_ID,
            TO_VARIANT(FST_NAME) AS FST_NAME,
            TO_VARIANT(MDL_NAME) AS MDL_NAME,
            TO_VARIANT(LST_NAME) AS LST_NAME,
            TO_VARIANT("PT_AGE") AS PT_AGE,
            TO_VARIANT(PT_ZIP) AS PT_ZIP,
            TO_VARIANT(ICD_CD) AS ICD_CD,
            TO_VARIANT(CPT_CD) AS CPT_CD,
            TO_VARIANT("ICDCD_NUMCODED") AS ICDCD_NUMCODED,
            TO_VARIANT("SUBCD_NBR") AS SUBCD_NBR,
            TO_VARIANT(OMC_ICD_RISK_NBR) AS OMC_ICD_RISK_NBR,
            TO_VARIANT("PAID_AMT_prev") AS PAID_AMT_PREV,
            TO_VARIANT("PAID_AMT_nxtyr") AS PAID_AMT_NXTYR,
            TO_VARIANT("clm_diff_col") AS CLM_DIFF_COL
            FROM {StoredTable}"""
        
        df = session.sql(sql_query)
        df.collect()
        
        trvalData_pd = pd.read_snowflake(f"{database_name}.{schema_name}.{CleanedTable}")
        trvalData_pd.fillna(0, inplace=True)
        return trvalData_pd

In [ ]:
import snowflake.snowpark.modin.plugin
import snowflake.snowpark.functions as F
from snowflake.ml.model import model_signature
from snowflake.ml.model.model_signature import ModelSignature
from snowflake.ml.model.model_signature import FeatureSpec
from snowflake.ml.model.model_signature import DataType
from snowflake.snowpark.functions import col, lit, typeof
from snowflake.snowpark.types import IntegerType, StringType
from snowflake.ml.modeling.metrics import mean_squared_error

import modin.pandas as pd


#from snowflake import telemetry
def train_val():
    print(strt_yr, last_yr)
    for stage in stages:
        print(stage)
        if stage =='Train':
            results= session.sql(f"SHOW TABLES LIKE 'TRAINING_TABLE' IN {database_name}.{schema_name}").collect()
            
            if len(results)>0:
                # file exists
                print("Training table exists")
                #trvalData = pd.read_snowflake(f"{database_name}.{schema_name}.TRAINING_TABLE")
            else:
                print("Training dataprep needs to be done")  
                train_val_dataprep(strt_yr, last_yr-1)
    
            TableName = "TRAINING_TABLE"
            NewTableName = "TRAINING_DATA"
     
        else:
            results= session.sql(f"SHOW TABLES LIKE 'VALIDATING_TABLE' IN {database_name}.{schema_name}").collect()
            
            if len(results)>0:
                # file exists
                print("Validating data table exists")
                #trvalData = pd.read_snowflake(f"{database_name}.{schema_name}.VALIDATING_TABLE")
            else:
                print("Validation dataprep needs to be done")  
                train_val_dataprep(int(last_yr)-1, int(last_yr))
            TableName = "VALIDATING_TABLE"
            NewTableName = "VALIDATING_DATA"
        #trvalData.columns = trvalData.columns.str.upper()
        #print(trvalData.dtypes)

        """Load the training/validaiton data from the default database, schema """
        trvalData = load_table(TableName, NewTableName)
        
        trvalData['MBR_ID'] = trvalData['MBR_ID'].astype(str)
        trvalData['FST_NAME'] = trvalData['FST_NAME'].astype(str)
        trvalData['MDL_NAME'] = trvalData['MDL_NAME'].astype(str)
        trvalData['LST_NAME'] = trvalData['LST_NAME'].astype(str)
        trvalData['OMC_ICD_RISK_NBR'] = trvalData['OMC_ICD_RISK_NBR'].astype(int)
        trvalData['ICDCD_NUMCODED'] = trvalData['ICDCD_NUMCODED'].astype(float)
        trvalData['SUBCD_NBR'] = trvalData['SUBCD_NBR'].astype(float)
        trvalData['PT_AGE'] = trvalData['PT_AGE'].astype(int)
        trvalData['PT_ZIP'] = trvalData['PT_ZIP'].astype(float)
        trvalData['CLM_DIFF_COL'] = trvalData['CLM_DIFF_COL'].astype(float)
        
        print(trvalData.dtypes)
        # Specify target columns to check for non-numeric values
            # convert those cell values to na and delete them
        feature_cols = ['PT_AGE','PT_ZIP','ICDCD_NUMCODED', 'OMC_ICD_RISK_NBR','SUBCD_NBR']
        target_col = ['CLM_DIFF_COL']
        metadata = ["MBR_ID", "FST_NAME", "MDL_NAME", 'LST_NAME','ICD_CD','CPT_CD','PAID_AMT_PREV']
        numeric_cols = ['OMC_ICD_RISK_NBR', 'PAID_AMT_PREV','SUBCD_NBR','ICDCD_NUMCODED']
        # Specify your exact casing for your XGBoost features and the Monotonicity target
        exact_feature_names = ["PT_AGE","PT_ZIP","ICDCD_NUMCODED", "OMC_ICD_RISK_NBR","SUBCD_NBR","CLM_DIFF_COL"]

        # Apply F.col("ORIGINAL_CASE") to keep your names in their original case
        selected_cols = [F.col(f'"{col}"').as_(col) for col in exact_feature_names]
        # Get the training data length y
        #NData = x_pt_age.shape[0]
        trvalData_snowpark = trvalData.to_snowpark(index=False)
        
        column_names = trvalData_snowpark.columns
        print("trvalData_snowpark columns")
        print(column_names)
        final_snowpark_df = trvalData_snowpark.select(selected_cols)
        
        ALL_COLS = ['MBR_ID', 'FST_NAME', 'MDL_NAME', 'LST_NAME','PT_AGE',\
            'PT_ZIP', 'ICD_CD', 'CPT_CD','ICDCD_NUMCODED', 'OMC_ICD_RISK_NBR',\
                'SUBCD_NBR','PAID_AMT_PREV','CLM_DIFF_COL']
        """
        # 2. Check each column to see if it contains the exact offending string
        bad_value = '8ND8NT9FW50'
        print("Scanning columns for the problematic value...")
        
        for col_name in ALL_COLS:
            # Filter rows where the column matches our string
            match_count = final_snowpark_df.filter(F.col(col_name).cast("string") == bad_value).count()
            
            if match_count > 0:
                print(f" Found it! Column '{col_name}' contains {match_count} instance(s) of '{bad_value}'.")
                
        """
        """
        #trvalData_snowpark = trvalData_snowpark.withColumn("OMC_ICD_RISK_NBR", col("OMC_ICD_RISK_NBR").astype("int"))
        trvalData_snowpark = trvalData_snowpark.with_column("OMC_ICD_RISK_NBR", F.col("OMC_ICD_RISK_NBR").cast(IntegerType()))
        trvalData_snowpark = trvalData_snowpark.with_column("PT_AGE", F.col("PT_AGE").cast(IntegerType()))
        column_names = trvalData_snowpark.columns
        
        print("trvalData_snowpark columns")
        
        print(column_names)
        """

        print(trvalData_snowpark.dtypes)
        trvalData = trvalData.astype({'MBR_ID':str,'FST_NAME':str, 'MDL_NAME': str,'LST_NAME':str})
        
        if stage == "Train":
            X_predict = trvalData[ALL_COLS].to_pandas()
            X_predict = X_predict.reset_index()
            print(f"X_predict pandas df cols: {list(X_predict.columns.values)}")
            
            # Initialize the model with hyperparameters
            xgb_model = XGBRegressor(
                enable_categorical=True,
                tree_method="hist",  # 'hist' or 'approx' required for categoricals
                n_estimators=1000,
                learning_rate=0.001,
                max_depth=5,
                input_cols = feature_cols, # specify feature columns
                label_cols = target_col,                        # Specify your continuous target
                passthrough_cols = metadata,
                output_cols=['CLMDIFF_PREDICTIONS']                # Specify the prediction output column
            )
            # Fit the XGB regressor model locally using pandas DataFrame
            # (avoids stored procedure that fails with 'No module named pandas')
            print(f"Training the XGB Regressor model")
            trvalData_pandas_fit = trvalData_snowpark.to_pandas()
            xgb_model.fit(trvalData_pandas_fit)
            train_features = trvalData_snowpark.first(10)
            """
            sig = model_signature.infer_signature(
                train_features,
                train_labels,
                input_feature_names=['PT_AGE','PT_ZIP','ICDCD_NUMCODED', 'OMC_ICD_RISK_NBR','SUBCD_NBR'],
                output_feature_names=['CLM_DIFF_COL'])
                """
            # Log model using the timestamp as the version name
            predictions_df = xgb_model.predict(X_predict)
            predictions_df_snowpark = session.create_dataframe(predictions_df)
            print(f"Training completed and saving the model")   
            rmse = mean_squared_error(
                df=predictions_df_snowpark,
                y_true_col_names="CLM_DIFF_COL",
                y_pred_col_names="CLMDIFF_PREDICTIONS",
                squared=False  # Setting this to False returns RMSE instead of MSE
            )
            print(f"Root Mean Squared Error: {rmse}")

            save_model(xgb_model,"XGBRegressor_clmsPredmodel_0171",trvalData,feature_cols, target_col,rmse,True)

            
                        
        else:
            print(stage)
            trvalData_pandas = trvalData[ALL_COLS].to_pandas()
            model = retrieve_model("XGBRegressor_clmsPredmodel_0171")
            if model is None:
                print("Error: Trained model not found")
                continue 
            
            # 4. Generate predictions using the .run() method
            # Pass your pandas or Snowpark DataFrame containing the input features
            predictions_df_snowpark = model.run(
                trvalData_snowpark.select("MBR_ID","PT_AGE","PT_ZIP","ICDCD_NUMCODED", "OMC_ICD_RISK_NBR","SUBCD_NBR", "PAID_AMT_PREV",'CLM_DIFF_COL'), 
                function_name="predict"
            )
            print(predictions_df.columns)

            
            #predictions_df_snowpark = trvalData_snowpark.join(predictions_df, on='MBR_ID')
               
        # Cast the DataFrame column to a String before performing your DML write
        #predictions_df_snowpark = predictions_df_snowpark.with_column("MBR_ID", col("MBR_ID").cast(StringType()))
        predictions_df_snowpark = predictions_df_snowpark.with_column("PRED_CLM_CHRG", predictions_df_snowpark['PAID_AMT_PREV'] + predictions_df_snowpark['CLMDIFF_PREDICTIONS'])
        # Delete some unnecessary columns
        predictions_df_snowpark = predictions_df_snowpark.drop("ICDCD_NUMCODED", "SUBCD_NBR","CLM_DIFF_COL","CLMDIFF_PREDICTIONS")
        # Select columns and rename them to be strictly alphanumeric (plus underscores)
        
        predictions_df_snowpark.write.mode("overwrite").save_as_table(f"{database_name}.{schema_name}.actual_pred_Table_{stage}ing_stage")
        #predictions_df_snowpark.to_snowflake(f"{database_name}.{schema_name}.actual_pred_Table_{stage}ing_stage", if_exists='replace', index=False)
        
    

In [ ]:
train_val()